# Weight Initialization & Training Dynamics

Companion notebook for the [Weight Initialization lesson](https://ml-viz-ruby.vercel.app/courses/neural-networks/06-weight-initialization).

We propagate a signal through a deep network and watch initialization decide whether activations
**vanish, explode, or stay stable** — then verify that **He initialization** preserves variance
through a ReLU stack where naive init fails. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## Intuition — the starting point decides if training even begins

Before a network learns anything, its weights are random — and *how* random matters enormously. Send
a signal through many layers and the activations' scale compounds: too-large weights make it
**explode**, too-small make it **vanish**, and either way the gradients die and training stalls
before it starts. The fix is to pick the initialization **variance** so that signal scale is
preserved layer to layer. **He initialization** (`Var = 2/n_in`) does this for ReLU nets; **Xavier**
(`2/(n_in+n_out)`) for tanh. A related trick, **learning-rate warmup**, protects the fragile first
steps. This notebook watches init decide survival-through-depth and validates He against `jax`.

## 1 — Signal variance across depth depends on init scale

Push a random input through 30 linear+ReLU layers and track the activation standard deviation per
layer. Too-large init explodes it; too-small vanishes it; the right scale keeps it ~constant.

In [ ]:
def propagate(scale, depth=30, width=256, seed=0):
    r = np.random.default_rng(seed)
    h = r.normal(size=width)
    stds = [h.std()]
    for _ in range(depth):
        W = r.normal(size=(width, width)) * scale
        h = np.maximum(W @ h, 0)            # linear + ReLU
        stds.append(h.std())
    return stds

fig, ax = plt.subplots(figsize=(8, 4.2))
for scale, name, c in [(0.02, 'too small (vanish)', '#fb7185'),
                       (0.20, 'too large (explode)', '#eab308'),
                       (np.sqrt(2/256), 'He init (stable)', '#2dd4bf')]:
    ax.plot(propagate(scale), label=f'{name}, scale={scale:.3f}', color=c)
ax.set_yscale('log'); ax.set_xlabel('layer'); ax.set_ylabel('activation std (log)')
ax.set_title('Initialization scale decides whether the signal survives depth')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**What to notice:** on the log axis, **too-small** init sends the activation std crashing toward 0
(vanishing signal) and **too-large** sends it rocketing up (exploding) within a few layers — both
fatal for a 30-layer net. **He init** keeps the std roughly flat across all 30 layers. The
initialization scale, alone, is the difference between a trainable and an untrainable network.

## 2 — He initialization preserves variance through ReLU

He init sets Var(W) = 2/n_in. The factor of 2 compensates for ReLU zeroing half its inputs. We
measure the per-layer variance ratio — it stays near 1 for He, but decays for the naive 1/n_in.

In [ ]:
def he_std(n_in):     return np.sqrt(2.0 / n_in)
def xavier_std(n_in, n_out): return np.sqrt(2.0 / (n_in + n_out))

width = 512
for name, scale in [('He (2/n)', he_std(width)), ('naive (1/n)', np.sqrt(1.0/width))]:
    stds = propagate(scale, depth=20, width=width, seed=3)
    print(f'{name:12s}: activation std at layer 1 = {stds[1]:.3f}, at layer 20 = {stds[20]:.3f}')
print('\nHe keeps the std roughly flat; the naive 1/n scale lets it decay with depth.')

**What to notice:** He's factor of **2** exactly compensates for ReLU zeroing half its inputs, so
the activation std stays near its layer-1 value even at layer 20. The naive `1/n` scale (no factor of
2) lets the std **decay** with depth — the extra 2× is precisely the ReLU correction.

## 3 — Learning-rate warmup

Warmup ramps the LR up over the first steps (when weights/optimizer stats are unreliable), then
decays it. We plot a linear-warmup + cosine-decay schedule.

In [ ]:
def lr_schedule(step, base=1e-3, warmup=200, total=2000):
    if step < warmup:
        return base * step / warmup                              # linear warmup
    prog = (step - warmup) / (total - warmup)
    return base * 0.5 * (1 + np.cos(np.pi * prog))               # cosine decay

steps = np.arange(2000)
lrs = [lr_schedule(s) for s in steps]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(steps, lrs, color='#818cf8')
ax.axvline(200, ls='--', color='#888', label='end of warmup')
ax.set_xlabel('training step'); ax.set_ylabel('learning rate')
ax.set_title('Linear warmup then cosine decay'); ax.legend(facecolor='#1a1d27', edgecolor='#444')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

**What to notice:** the schedule ramps the LR up **linearly** over the first 200 steps, then
**cosine-decays** it. Early on, weights are random and (for Adam) the moment estimates are unreliable,
so a big step can destabilize training; warmup eases in until the statistics settle. This is standard
for training transformers.

## The library way — validate He against `jax`

Frameworks ship these initializers (`torch.nn.init.kaiming_normal_`,
`jax.nn.initializers.he_normal`). The cell samples a He-initialized weight matrix with `jax` and
confirms its standard deviation matches our `√(2/n_in)` formula.

In [ ]:
import jax, jax.nn.initializers as jinit

he = jinit.he_normal()                                   # jax's He/Kaiming init
W = np.array(he(jax.random.PRNGKey(0), (512, 256)))      # (fan_in=512, fan_out=256)
print(f'jax he_normal std: {W.std():.5f}')
print(f'our he_std(512):   {he_std(512):.5f}')
assert abs(W.std() - he_std(512)) < 5e-3, "jax He init must match sqrt(2/n_in)"
print('\njax.nn.initializers.he_normal == our sqrt(2/n_in) ✓')

**What to notice:** `jax`'s built-in He initializer produces weights with exactly the
`√(2/n_in)` standard deviation we derived — the library isn't doing anything mysterious, just applying
the variance-preservation rule. `torch.nn.init.kaiming_normal_` is the same thing.

## Gotchas & tradeoffs

- **Never initialize all weights to the same value** (e.g. zero). Every neuron in a layer would
  compute the same thing and receive the same gradient — **symmetry never breaks** and the layer acts
  like a single neuron forever. Randomness is essential.
- **Match the init to the activation.** He (`2/n_in`) for ReLU/variants; Xavier/Glorot
  (`2/(n_in+n_out)`) for tanh/sigmoid. Using the wrong one reintroduces vanishing/exploding.
- **Init only sets the starting point.** BatchNorm/LayerNorm and residual connections make deep nets
  far more robust to init choice than plain stacks.
- **Warmup matters most for adaptive optimizers & transformers**, where early large steps on
  unreliable statistics can diverge.

In [ ]:
# Zero (or constant) init: every neuron is identical -> symmetry never breaks
W_zero = np.zeros((4, 3)); x = np.array([1.0, 2.0, 3.0])
print('zero-init pre-activations:', W_zero @ x, '-> all identical across neurons')

# He (ReLU) vs Xavier (tanh): each keeps its own activation's variance near 1
np.random.seed(0)
h_relu = np.maximum(np.random.randn(4096, 256) @ (np.random.randn(256) * he_std(256)), 0)
print(f'ReLU + He: activation var = {h_relu.var():.3f}  (kept O(1) by the 2/n_in factor, not vanishing)')

**What to notice:** zero-initialized neurons all produce the identical pre-activation, so they can
never differentiate — the reason initialization must be **random**. And He-scaled weights keep the
post-ReLU activation variance **O(1)** (here ~0.73) instead of collapsing toward zero, exactly what
the `2/n_in` factor is designed to do.

## Key takeaways

- Initialization scale decides whether a deep signal **vanishes, explodes, or stays stable** — and
  thus whether training can even start.
- **He** (`Var = 2/n_in`) preserves variance through **ReLU**; **Xavier** (`2/(n_in+n_out)`) through
  **tanh/sigmoid** — the factor of 2 is the ReLU correction.
- Weights must be **random** (never all-equal) to break symmetry between neurons.
- **LR warmup** protects the fragile first steps (especially for Adam/transformers); frameworks'
  `kaiming_normal_` / `he_normal` apply exactly the `√(2/n_in)` rule.

**Next:** the [course quiz](https://ml-viz-ruby.vercel.app/courses/neural-networks/07-quiz).

## ✏️ Your turn

**Exercise.** Implement `he_init_std(n_in)` (= √(2/n_in)) and `warmup_lr(step, base, warmup)` — the
linear warmup phase that returns `base·step/warmup` while ramping up and `base` once `step ≥ warmup`.

In [ ]:
def he_init_std(n_in):
    # TODO(you): standard deviation for He initialization
    return ...

def warmup_lr(step, base=1e-3, warmup=200):
    # TODO(you): base*step/warmup while step < warmup, else base
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(he_init_std(2), 1.0)                  # sqrt(2/2) = 1
assert np.isclose(he_init_std(512), np.sqrt(2/512))
assert warmup_lr(0) == 0.0                               # LR starts at 0
assert np.isclose(warmup_lr(100, 1e-3, 200), 5e-4)      # halfway through warmup
assert warmup_lr(500, 1e-3, 200) == 1e-3               # past warmup -> base LR
print('\u2713 He init scale and warmup schedule are correct')

<details>
<summary>Solution</summary>

```python
def he_init_std(n_in):
    return np.sqrt(2.0 / n_in)

def warmup_lr(step, base=1e-3, warmup=200):
    return base * step / warmup if step < warmup else base
```

He's factor of 2 exactly cancels the variance ReLU loses by zeroing half its inputs, keeping signal
stable through depth. Warmup protects the fragile first steps, when weights and adaptive-optimizer
statistics haven't settled.

</details>